In [17]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List , Dict , Any , Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [18]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: FYP_Proposal_AI_University_Management_System.pdf


  ✓ Loaded 13 pages

Total documents loaded: 13


In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks = text_splitter.split_documents(all_pdf_documents)

print(f"Total pages/documents: {len(all_pdf_documents)}")
print(f"Total chunks created: {len(chunks)}")


Total pages/documents: 13
Total chunks created: 33


In [13]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1724.74it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\lenovo\AppData\Local\Temp\ipykernel_3580\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


VectorStore

In [14]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [15]:
chunks

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-02T10:30:37+00:00', 'title': 'Final Year Report Template', 'author': 'Barry Smyth', 'moddate': '2026-09-02T10:30:37+00:00', 'source': '..\\data\\pdf\\FYP_Proposal_AI_University_Management_System.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'source_file': 'FYP_Proposal_AI_University_Management_System.pdf', 'file_type': 'pdf'}, page_content='AI Powered University Management System \nFinal Year Project Proposal \nSession 2023-2027 \n \n \nA project submitted in partial fulfilment of the requirements for the \nDegree \nof  \nBS in Computer Science / Software Engineering / Artificial Intelligence \n \n \n \nDepartment of Computer Science \nCOMSATS University Islamabad (CUI), Lahore Campus'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-02T10:30:37+00:00', 'title': 'Final Year Report Template', 'author': '

In [16]:
#convert the text to embeddings

texts = [doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)


## store in the vector database
vectorstore.add_documents(chunks , embeddings)

Generating embeddings for 33 texts...


Batches: 100%|██████████| 2/2 [00:04<00:00,  2.47s/it]


Generated embeddings with shape: (33, 384)
Adding 33 documents to vector store...
Successfully added 33 documents to vector store
Total documents in collection: 33


### Retriever Pipeline From VectorStore

In [25]:

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        max_distance: float = None
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: Search query
            top_k: Number of results to retrieve
            max_distance: Optional maximum allowed distance.
                          Lower distance = more similar.

        Returns:
            List of dictionaries containing documents and metadata.
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Max distance: {max_distance}")

        # 1. Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        try:
            # 2. Search ChromaDB
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            # 3. Check if results exist
            if not results["documents"] or not results["documents"][0]:
                print("No documents found")
                return []

            documents = results["documents"][0]
            metadatas = results["metadatas"][0]
            distances = results["distances"][0]
            ids = results["ids"][0]

            # 4. Process results
            for rank, (doc_id, document, metadata, distance) in enumerate(
                zip(ids, documents, metadatas, distances),
                start=1
            ):

                # Lower distance means better match
                if max_distance is not None and distance > max_distance:
                    continue

                retrieved_docs.append({
                    "id": doc_id,
                    "content": document,
                    "metadata": metadata,
                    "distance": distance,
                    "rank": rank
                })

            print(
                f"Retrieved {len(retrieved_docs)} documents "
                f"(after filtering)"
            )

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Create retriever
rag_retriever = RAGRetriever(
    vectorstore,
    embedding_manager
)


In [26]:
rag_retriever

In [27]:
rag_retriever.retrieve("What is the abstract of project")

Retrieving documents for query: 'What is the abstract of project'
Top K: 5, Max distance: None
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_a42e9017_2',
  'content': 'Project ID (for office use)   \nType of project [  ] Traditional              [  ] Industrial  [✓ ] Continuing \nNature of project [✓ ] Development            [  ] Research & Development  \nSustainable Development \nGoals(SDGs) \n[  ] Good Health and Well-Being                      [✓ ] Quality Education  \n[✓ ]  Industry, Innovation, and Infrastructure     [  ] Gender Equality \n[  ] Decent Work and Economic Growth           [  ]  Climate Action \nArea of specialization \n[✓ ] Artificial Intelligence (AI)      [  ] Blockchain               [  ] Cybersecurity \n[  ] Data Science and Analytics    [  ] Game Development \n[  ] Internet of Things (IoT)          [✓ ]  Natural Language Processing (NLP) \n[  ] Mobile App Development      [✓ ]  Web Development \nProject Group Members \nSr.# Reg. # Student Name Email ID Phone # Signature \n(i) \nGroup Leader \nSP23-BSE-135 \nMuhammad Zaid Amjad sp23-bse-135@cuilahore.edu.pk 03100044108  \n(ii) SP23-BSE-00

In [23]:
print(vectorstore.collection.count())

33


In [24]:
query = "What is the abstract of project"

query_embedding = embedding_manager.generate_embeddings([query])[0]

results = vectorstore.collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

print(results)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.97it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_a42e9017_2', 'doc_fd412312_22', 'doc_d36d65f6_0', 'doc_397bcfc0_5', 'doc_216e0b52_26']], 'embeddings': None, 'documents': [['Project ID (for office use)   \nType of project [  ] Traditional              [  ] Industrial  [✓ ] Continuing \nNature of project [✓ ] Development            [  ] Research & Development  \nSustainable Development \nGoals(SDGs) \n[  ] Good Health and Well-Being                      [✓ ] Quality Education  \n[✓ ]  Industry, Innovation, and Infrastructure     [  ] Gender Equality \n[  ] Decent Work and Economic Growth           [  ]  Climate Action \nArea of specialization \n[✓ ] Artificial Intelligence (AI)      [  ] Blockchain               [  ] Cybersecurity \n[  ] Data Science and Analytics    [  ] Game Development \n[  ] Internet of Things (IoT)          [✓ ]  Natural Language Processing (NLP) \n[  ] Mobile App Development      [✓ ]  Web Development \nProject Group Members \nSr.# Reg. # Student Name Emai